In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib

In [2]:
import set_paths
import src.preprocesing_tools as pt

In [3]:
raw_data = pd.read_csv(r"..\data\raw.csv")
print("samples:", raw_data.shape[0], "features:", raw_data.shape[1]-1)
raw_data.head()
#Validamos que no haya NaN's
print("¿Hay algún NaN? ->", raw_data.isna().any().unique()[0])
raw_data.head(10)

samples: 6819 features: 95
¿Hay algún NaN? -> False


,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,1,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,...,0.716845,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469
1,1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,...,0.795297,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794
2,1,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,...,0.774670,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474
3,1,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,...,0.739555,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982
4,1,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,...,0.795016,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490
5,1,0.388680,0.415177,0.419134,0.590171,0.590251,0.998758,0.796903,0.808771,0.303116,...,0.710420,0.005278,0.622605,0.590172,0.829939,0.285087,0.026675,0.564538,1,0.019534
6,0,0.390923,0.445704,0.436158,0.619950,0.619950,0.998993,0.797012,0.808960,0.302814,...,0.736619,0.018372,0.623655,0.619949,0.829980,0.292504,0.026622,0.564200,1,0.015663
7,0,0.508361,0.570922,0.559077,0.601738,0.601717,0.999009,0.797449,0.809362,0.303545,...,0.815350,0.010005,0.623843,0.601739,0.841459,0.278607,0.027031,0.566089,1,0.034889
8,0,0.488519,0.545137,0.543284,0.603612,0.603612,0.998961,0.797414,0.809338,0.303584,...,0.803647,0.000824,0.623977,0.603613,0.840487,0.276423,0.026891,0.565592,1,0.065826
9,0,0.495686,0.550916,0.542963,0.599209,0.599209,0.999001,0.797404,0.809320,0.303483,...,0.804195,0.005798,0.623865,0.599205,0.840688,0.279388,0.027243,0.566668,1,0.030801


In [4]:
# Los datos son numéricos, por lo que verificamos que el tipo de dato sea siempre el adecuado
raw_data.dtypes.unique()

array([dtype('int64'), dtype('float64')], dtype=object)

In [5]:
#La columna "Net Income Flag", parece tener siempre el mismo valor (1), 
#Validamos que sea cierto y que no haya más casos de este tipo
same_value_columns = []
for column in raw_data.columns:
    if pt.has_the_same_value(raw_data, column):
        same_value_columns.append(column)
        print(f"{column} tiene el mismo valor en todas las muestras")
#Eliminamos las columnas que no aportan información por tener siempre el mismo valor
raw_data = raw_data.drop(same_value_columns, axis=1)

 Net Income Flag tiene el mismo valor en todas las muestras


In [6]:
#Verificamos que no haya filas repetidas
print("¿Hay filas duplicadas? ->", raw_data.duplicated().any())
#Verificamos que no haya columnas repetidas
print("¿Hay columnas duplicadas? ->", raw_data.T.duplicated().any())

¿Hay filas duplicadas? -> False
¿Hay columnas duplicadas? -> True


In [7]:
#Rastreamos las columnas duplicadas:
index_dup_columns = list(pt.get_duplicated_index(raw_data.T))
print("Índices de columnas duplicadas:", index_dup_columns)
raw_data.iloc[:, index_dup_columns]

Índices de columnas duplicadas: [np.int64(64), np.int64(66), np.int64(77), np.int64(78)]


,Current Liabilities/Liability,Current Liabilities/Equity,Current Liability to Liability,Current Liability to Equity
0,0.676269,0.339077,0.676269,0.339077
1,0.308589,0.329740,0.308589,0.329740
2,0.446027,0.334777,0.446027,0.334777
3,0.615848,0.331509,0.615848,0.331509
4,0.975007,0.330726,0.975007,0.330726
...,...,...,...,...
6814,0.786888,0.330914,0.786888,0.330914
6815,0.849898,0.329753,0.849898,0.329753
6816,0.553964,0.326921,0.553964,0.326921
6817,0.893241,0.329294,0.893241,0.329294


In [8]:
#Eliminamos los duplicados de cada columna
preprocesed_data = raw_data.drop(raw_data.columns[index_dup_columns[2:]], axis=1)
#Guardamos el dataset preprocesado
preprocesed_data.to_csv(r"..\data\preprocessed.csv", index=False)